# Your first scraper
In this project, we will guide you step by step through the process of:

1. creating a self-contained development environment.
1. retrieving some information from an API (a website for computers)
2. leveraging it to scrape a website that does not provide an API
3. saving the output for later processing

Here we query an API for a list of countries and their past leaders. We then extract and sanitize their short bio from Wikipedia. Finally, we save the data to disk.

This task is often the first (coding) step of a datascience project and you will often come back to it in the future.

You will study topics such as *scraping*, *data structures*, *regular expressions*, *concurrency* and *file handling*. We will point out useful resources at the appropriate time. 

Let's dive in!

## 0. Creating a clean environment

Use the [`venv`](https://docs.python.org/3/library/venv.html) command to create a new environment called `wikipedia_scraper_env`.

Activate it and add it to you `.gitignore` file. 

You will find more info about virtual environments in the course content and on the web.

## 1. API Scraping

### 1a. A simple API query
You will start with the basics: how to do a simple request to an [API endpoint](../../2.python/2.python_advanced/05.Scraping/5.apis.ipynb).

You will use the [requests](https://requests.readthedocs.io/en/latest/) external library through the `import` keyword. NOTE: external libraries need to be installed first. Check the [request Quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/) section of the documentation to:

1. Use the `get()` method to connect to this endpoint: https://country-leaders.onrender.com/status
2. Check if the `status_code` is equal to 200, which means OK.
    * if OK, `print()` the `text`` of the response.
    * if not, `print()` the `status_code`. 

Here is an explanation of [HTTP status codes](https://en.wikipedia.org/wiki/List_of_HTTP_status_codes).


In [8]:
# import the requests library (1 line)
import requests #à installer sur venv

# assign the root url (without /status) to the root_url variable for ease of reference (1 line)
root_url = "https://country-leaders.onrender.com"

# assign the /status endpoint to another variable called status_url (1 line)
status_url = root_url + "/status"

# query the /status endpoint using the get() method and store it in the req variable (1 line)
req = requests.get(status_url)

# check the status_code using a condition and print appropriate messages (4 lines)
if req.status_code == 200:
    print(req.text)
else:
    print(req.status_code)

"Alive"


### 1b. Dealing with JSON

[JSON](https://quickref.me/json) is the preferred format to deal with data over the web. You cannot avoid it so you would better get acquainted.

Connect to another endpoint called `/countries` but this time the API will return data in the JSON format. 


In [9]:
# Set the countries_url variable (1 line)
countries_url = root_url + "/countries"

# query the /countries endpoint using the get() method and store it in the req variable (1 line)
req = requests.get(countries_url)

# Get the JSON content and store it in the countries variable (1 line)
countries = req.json()

# display the request's status code and the countries variable (1 line)
print("request's status code : ", req.status_code,".", "And countries : ", countries)

request's status code :  403 . And countries :  {'message': 'The cookie is missing'}


### 1c. Cookies anyone?

It looks like the access to this API is restricted...
Query the `/cookie` endpoint and extract the appropriate field to access your cookie.

You will need to use this cookie in each of the following API requests.

In [10]:
# Set the cookie_url variable (1 line)
cookie_url = root_url + "/cookie"

# Query the endpoint, set the cookies variable and display it (2 lines)
req = requests.get(cookie_url) 
cookies = req.cookies
"""req contient toute la réponse du serveur : code de statut, en-têtes, contenu et cookies
req.status_code renvoit le code HTTP (ex : 200, 404, 403)
req.text renvoit le contenu texte
req.json() renvoit le contenu JSON
req.headers renvoit les en-têtes HTTP
req.cookies renvoit les cookies renvoyés par le serveur

dans mon cas l'endpoint /cookie est justement conçu pour envoyer un cookie. il faut donc le récupérer dans l'attribut cookies de la réponse"""

print(cookies)

<RequestsCookieJar[<Cookie user_cookie=596ccbc0-d03f-48bd-84e5-404a4b17e343 for country-leaders.onrender.com/>]>


Try to query the countries endpoint using the cookie, save the output and print it.

In [11]:
# query the /countries endpoint, assign the output to the countries variable (1 line)
countries = requests.get(countries_url, cookies=cookies).json()
"""cookies=cookies est le paramètre qui envoie le cookie récupéré précédemment dans le serveur"""

# display the countries variable (1 line)
print(countries)

['fr', 'us', 'be', 'ma', 'ru']


Étape 1 : Tu demandes un badge à l'accueil
req = requests.get(cookie_url)
cookies = req.cookies

C'est comme si l'accueil te donnait un badge d'accès.

req = tout ce que l'accueil te remet (documents, infos, badge, etc.)
req.cookies = le badge uniquement
cookies = req.cookies = tu mets le badge dans ta poche pour le réutiliser
Étape 2 : Tu veux entrer dans le parking

Si tu fais :

requests.get(countries_url)

c'est comme arriver à la barrière sans badge.

Le gardien répond :

403 Forbidden
The cookie is missing

👉 « Je ne vous connais pas, accès refusé. »

Étape 3 : Tu présentes ton badge
requests.get(countries_url, cookies=cookies)

C'est comme montrer le badge que tu as récupéré à l'accueil.

Le gardien vérifie :

Badge valide ? Oui.

et ouvre la barrière.

Chances are the cookie has expired... Thanksfully, you got a nice error message. For now, simply execute the last 2 cells quickly so you get a result.

Autre exemple avec une voiture :

plaque = "1-ABC-123"

Puis :

entrer_parking(plaque=plaque)

équivaut à :

entrer_parking(plaque="1-ABC-123")

Le premier plaque= est le nom du paramètre de la fonction, le second plaque est ta variable.

### 1d. Getting the actual data from the API

Query the `/leaders` endpoint.

In [12]:
# Set the leaders_url variable (1 line)
leaders_url = root_url + "/leaders"

# query the /leaders endpoint, assign the output to the leaders variable (1 line)
leaders = requests.get(leaders_url, cookies = cookies).json()

# display the leaders variable (1 line)
print(leaders)

{'message': 'Please specify a country'}


It looks like this endpoint requires additional information in order to return its result. Check the API [*documentation*](https://country-leaders.onrender.com/docs) in your web browser.

Change the query to accept *parameters*. You should know where to find help by now.

In [13]:
# query the /leaders endpoint using cookies and parameters (take any country in countries)
req = requests.get(leaders_url, cookies=cookies,params={"country": "be"})

# assign the output to the leaders variable (1 line)
leaders = req.json()

# display the leaders variable (1 line)
import json; print(json.dumps(leaders, indent= 4))

[
    {
        "id": "Q12978",
        "first_name": "Guy",
        "last_name": "Verhofstadt",
        "birth_date": "1953-04-11",
        "death_date": null,
        "place_of_birth": "Dendermonde",
        "wikipedia_url": "https://nl.wikipedia.org/wiki/Guy_Verhofstadt",
        "start_mandate": "1999-07-12",
        "end_mandate": "2008-03-20"
    },
    {
        "id": "Q12981",
        "first_name": "Yves",
        "last_name": "Leterme",
        "birth_date": "1960-10-06",
        "death_date": null,
        "place_of_birth": "Wervik",
        "wikipedia_url": "https://nl.wikipedia.org/wiki/Yves_Leterme",
        "start_mandate": "2009-11-25",
        "end_mandate": "2011-12-06"
    },
    {
        "id": "Q12983",
        "first_name": "Herman",
        "last_name": "None",
        "birth_date": "1947-10-31",
        "death_date": null,
        "place_of_birth": "Etterbeek",
        "wikipedia_url": "https://nl.wikipedia.org/wiki/Herman_Van_Rompuy",
        "start_mandate": "2

### 1e. A sneak peak at the data (finally)

Look inside a few examples. Notice the dictionary keys available for each entry.
**je vois id, first_name, ..., end_mandate**

You have your first example of *structured data*. This data was sanitized for your benefit, meaning it is readily exploitable without modification.

You will also notice there is a Wikipedia link for each entry. You will need to extract additional information there. This will be a case of *semi-structured* data.

The /countries endpoint returns a `list` of several country codes.

You need to loop through this list and query the /leaders endpoint for each one. Save each `json` result in a dictionary called `leaders_per_country`.

In [14]:
# 4 lines
leaders_per_country = {}
for country in countries:
    #query the /leaders endpoint for each countrie
    req = requests.get(leaders_url, cookies= cookies, params= {"country" : country})
    leaders_per_country[country] = req.json()
import json; print(json.dumps(leaders_per_country, indent = 4))


{
    "fr": [
        {
            "id": "Q157",
            "first_name": "Fran\u00e7ois",
            "last_name": "Hollande",
            "birth_date": "1954-08-12",
            "death_date": null,
            "place_of_birth": "Rouen",
            "wikipedia_url": "https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande",
            "start_mandate": "2012-05-15",
            "end_mandate": "2017-05-14"
        },
        {
            "id": "Q329",
            "first_name": "Nicolas",
            "last_name": "Sarkozy",
            "birth_date": "1955-01-28",
            "death_date": null,
            "place_of_birth": "Paris",
            "wikipedia_url": "https://fr.wikipedia.org/wiki/Nicolas_Sarkozy",
            "start_mandate": "2007-05-16",
            "end_mandate": "2012-05-15"
        },
        {
            "id": "Q2038",
            "first_name": "Fran\u00e7ois",
            "last_name": "Mitterrand",
            "birth_date": "1916-10-26",
            "death_date": "19

In [15]:
# or 1 line
leaders_per_country = {country: requests.get(leaders_url, cookies=cookies, params={"country": country}).json() for country in countries}; import json; print(json.dumps(leaders_per_country, indent=4))

{
    "fr": [
        {
            "id": "Q157",
            "first_name": "Fran\u00e7ois",
            "last_name": "Hollande",
            "birth_date": "1954-08-12",
            "death_date": null,
            "place_of_birth": "Rouen",
            "wikipedia_url": "https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande",
            "start_mandate": "2012-05-15",
            "end_mandate": "2017-05-14"
        },
        {
            "id": "Q329",
            "first_name": "Nicolas",
            "last_name": "Sarkozy",
            "birth_date": "1955-01-28",
            "death_date": null,
            "place_of_birth": "Paris",
            "wikipedia_url": "https://fr.wikipedia.org/wiki/Nicolas_Sarkozy",
            "start_mandate": "2007-05-16",
            "end_mandate": "2012-05-15"
        },
        {
            "id": "Q2038",
            "first_name": "Fran\u00e7ois",
            "last_name": "Mitterrand",
            "birth_date": "1916-10-26",
            "death_date": "19

It is finally time to create a `get_leaders()` function for the above code. You will build on it later-on. This function takes no parameter. Inside it, you will need to:
1. define the urls
2. get the cookies
2. get the countries
3. loop over them and save their leaders in a dictionary
4. return the dictionary

In [45]:
# < 15 lines
import requests

def get_leaders():
    #define the urls
    root_url = "https://country-leaders.onrender.com"
    status_url = root_url + "/status"
    countries_url = root_url + "/countries"
    cookie_url = root_url + "/cookie"
    leaders_url = root_url + "/leaders"
    
    #get the cookies
    req1 = requests.get(cookie_url) 
    cookies = req1.cookies

    #get the countries
    countries = requests.get(countries_url, cookies=cookies).json()

    #loop over them and save their leaders in a dictionary
    data ={}
    for country in countries:
        req2 = requests.get(leaders_url, cookies= cookies, params= {"country" : country})
        data[country] = req2.json()
    return data

Test your function, save the result in the `leaders_per_country` dictionary and check its ouput.

In [46]:
# 2 lines
import json
leaders_per_country = get_leaders(); print(json.dumps(leaders_per_country,indent=4))

{
    "fr": [
        {
            "id": "Q157",
            "first_name": "Fran\u00e7ois",
            "last_name": "Hollande",
            "birth_date": "1954-08-12",
            "death_date": null,
            "place_of_birth": "Rouen",
            "wikipedia_url": "https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande",
            "start_mandate": "2012-05-15",
            "end_mandate": "2017-05-14"
        },
        {
            "id": "Q329",
            "first_name": "Nicolas",
            "last_name": "Sarkozy",
            "birth_date": "1955-01-28",
            "death_date": null,
            "place_of_birth": "Paris",
            "wikipedia_url": "https://fr.wikipedia.org/wiki/Nicolas_Sarkozy",
            "start_mandate": "2007-05-16",
            "end_mandate": "2012-05-15"
        },
        {
            "id": "Q2038",
            "first_name": "Fran\u00e7ois",
            "last_name": "Mitterrand",
            "birth_date": "1916-10-26",
            "death_date": "19

## 2. Extracting data from Wikipedia

Query one of the leaders' Wikipedia urls and display its `text` (not JSON).

In [47]:
# 3 lines
import requests
french_leader_url = "https://fr.wikipedia.org/wiki/Nicolas_Sarkozy"
headers = {"User-Agent": "Mozilla/5.0"} #à ajouter sinon pas d'accès
req = requests.get(french_leader_url, headers=headers)
print(req.text)

<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard" lang="fr" dir="ltr">
<head>
<meta charset="UTF-8">
<title>Nicolas Sarkozy — Wikipédia</title>
<script>(function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vect

Ouch! You get the raw HTML code of the webpage. If you try to deal with it without tools, you will be there all night. Instead, use the [beautiful soup 4](https://www.crummy.com/software/BeautifulSoup/bs4/doc/) *external* library. You will find more info about it [here](../../2.python/2.python_advanced/05.Scraping/1.beautifulsoup_basic.ipynb) and [here](../../2.python/2.python_advanced/05.Scraping/2.beautifulsoup_advanced.ipynb)

Using the Quickstart section, start by importing the library and loading the output of your `get_text()` function.

Use the `prettify()` function and print it to take a look. You will start the actual parsing in the next step.

In [19]:
# 3 lines
#from bs4 import BeautifulSoup; import requests

"""get_text est une méthode BeautifulSoup qui extrait le texte lisible d'une page HTML, en supprimant les balises"""
"""prettify est aussi une méthode BeautifulSoup qui affiche le HTML bien indenté et structuré, pour le rendre lisible"""

#soup = BeautifulSoup(req.text, 'html')
#print("get text : ",soup.get_text(), "prettify : ", soup.prettify())

'prettify est aussi une méthode BeautifulSoup qui affiche le HTML bien indenté et structuré, pour le rendre lisible'

In [20]:
from bs4 import BeautifulSoup; import requests
soup = BeautifulSoup(req.text, 'html')
print(soup.get_text())




Wikimedia Error






Error
Too many requests. Please respect our robot policy https://w.wiki/4wJS. (dd12474)


If you report this error to the Wikimedia System Administrators, please include the details below.Request served via cp3068 cp3068, Varnish XID 617981464Upstream caches: cp3068 intError: 403, Too many requests. Please respect our robot policy https://w.wiki/4wJS. (dd12474) at Thu, 04 Jun 2026 07:06:33 GMTSensitive client informationIP address: 78.29.192.45





In [21]:
from bs4 import BeautifulSoup; import requests
soup = BeautifulSoup(req.text, 'html')
print(soup.prettify())

<!DOCTYPE html>
<html lang="en">
 <meta charset="utf-8"/>
 <title>
  Wikimedia Error
 </title>
 <style>
  * { margin: 0; padding: 0; }
body { background: #fff; font: 15px/1.6 sans-serif; color: #333; }
.content { margin: 7% auto 0; padding: 2em 1em 1em; max-width: 640px; display: flex; flex-direction: row; flex-wrap: wrap; }
.footer { clear: both; margin-top: 14%; border-top: 1px solid #e5e5e5; background: #f9f9f9; padding: 2em 0; font-size: 0.8em; text-align: center; }
img { margin: 0 2em 2em 0; }
a img { border: 0; }
h1 { margin-top: 1em; font-size: 1.2em; }
.content-text { flex: 1; }
p { margin: 0.7em 0 1em 0; }
a { color: #0645ad; text-decoration: none; }
a:hover { text-decoration: underline; }
code { font-family: sans-serif; }
summary { font-weight: bold; cursor: pointer; }
details[open] { background: #970302; color: #dfdedd; }
.text-muted { color: #777; }
@media (prefers-color-scheme: dark) {
  a { color: #9e9eff; }
  body { background: transparent; color: #ddd; }
  .footer { bor

That looks better but you need to extract the right part of the webpage: the text of the first paragraph.

It is a bit tricky because Wikipedia pages slightly differ in structure from one language to the next. We cannot simply get the text for the first HTML paragraph.

You will start by getting all the HTML paragraphs from the HTML source and saving them in the `paragraphs` variable.

Use the documentation or google the appropriate keywords.

In [22]:
# 2 lines
#FAUX
#paragraphs = soup.select("#mw-content-text p")
#print(next(p.get_text() for p in paragraphs if p.get_text()))

If you try different urls, you might find that the paragraph you want may be at a different index each time.

That is where you need to be clever and ask yourself what would be a reliable way to identify the right index ie. which string matches only the first paragraph whatever the language...

Spend a good 30 minutes on the problem and brainstorm with your fellow learners. If you come out empty handed, ask your coach.

1. Loop over the HTML paragraphs
2. When you have identified the correct one:
   * Store the [text](https://www.crummy.com/software/BeautifulSoup/bs4/doc/#output) inside the `first_paragraph` variable
   * Exit the loop

In [23]:
# <10 lines
#paragraphs = soup.select("#mw-content-text p")


#first_paragraph = ""

#for p in paragraphs:
#    text = p.get_text().strip()
#    if text:
#        first_paragraph = text
#        break

#print(first_paragraph)

In [ ]:
#import requests
#french_leader_url = "https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande"
#headers = {"User-Agent": "Mozilla/5.0"} #à ajouter sinon pas d'accès
#req = requests.get(french_leader_url, headers=headers)

#from bs4 import BeautifulSoup
#soup = BeautifulSoup(req.text, 'html')

#paragraphs = soup.select("#mw-content-text > div > p")

#first_paragraph = ""
#for p in paragraphs:
#    text = p.get_text().strip()
#    if text:
#        first_paragraph = text
#        break

#print(first_paragraph)

François Hollande [fʁɑ̃swa ɔlɑ̃d][n 3] Écouterⓘ, né le 12 août 1954 à Rouen (Seine-Inférieure), est un haut fonctionnaire et homme d'État français. Il est président de la République française du 15 mai 2012 au 14 mai 2017.


At this stage, you can create a function to maintain consistency in your code. We will give you its *skeleton*, you will copy the code you wrote and make it work inside a function.

Don't forget to test your function.

In [ ]:
#for p in paragraphs:
#    if p.find("b"):
#        first_paragpraph = p.get.get_text()
#        break
#print(first_paragpraph)

NameError: name 'paragraphs' is not defined

In [48]:
# 10 lines
# def get_first_paragraph(wikipedia_url):
#   print(wikipedia_url) # keep this for the rest of the notebook
#   [insert your code]
#   return first_paragraph

def get_first_paragraph(wikipedia_url):

    print(wikipedia_url)
    headers = {"User-Agent": "Mozilla/5.0"}
    wiki_text = requests.get(wikipedia_url, headers = headers)
    soup = BeautifulSoup(wiki_text.text)
    paragraphs = soup.find_all("p")
    for p in paragraphs:
        if p.find("b"):
            first_paragraph = p.get_text()
        break

    return first_paragraph

In [50]:
# Test: 3 lines
wikipedia_url = leaders_per_country["us"][2]["wikipedia_url"]
first_paragraph = get_first_paragraph(wikipedia_url)
print(first_paragraph)

https://en.wikipedia.org/wiki/Abraham_Lincoln


UnboundLocalError: cannot access local variable 'first_paragraph' where it is not associated with a value

In [88]:
# < 15 lines
import requests

def get_leaders():
    #define the urls
    root_url = "https://country-leaders.onrender.com"
    status_url = root_url + "/status"
    countries_url = root_url + "/countries"
    cookie_url = root_url + "/cookie"
    leaders_url = root_url + "/leaders"
    
    #get the cookies
    req1 = requests.get(cookie_url) 
    cookies = req1.cookies

    #get the countries
    countries = requests.get(countries_url, cookies=cookies).json()

    #loop over them and save their leaders in a dictionary
    data ={}
    for country in countries:
        req2 = requests.get(leaders_url, cookies= cookies, params= {"country" : country})
        data[country] = req2.json()
    return data
#------
def get_first_paragraph(wikipedia_url):
    get_leaders()
    print(wikipedia_url)
    headers = {"User-Agent": "Mozilla/5.0"}
    wiki_text = requests.get(wikipedia_url, headers = headers)
    soup = BeautifulSoup(wiki_text.text)
    paragraphs = soup.find_all("p")
    for p in paragraphs:
        if p.find("b"):
            first_paragraph = p.get_text()
            break
    return first_paragraph

In [83]:
# Test: 3 lines
wikipedia_url = leaders_per_country["fr"][1]["wikipedia_url"]
first_paragraph = get_first_paragraph(wikipedia_url)
print(first_paragraph)

https://fr.wikipedia.org/wiki/Nicolas_Sarkozy
Nicolas Sarközy de Nagy-Bocsa, dit Nicolas Sarkozy (/ni.kɔ.la saʁ.kɔ.zi/ Écouterⓘ ; en hongrois Sárközy ou Sárközi ,,), né le 28 janvier 1955 à Paris 17e (Seine), est un homme d'État français. Il est président de la République française du 16 mai 2007 au 15 mai 2012.



### 2a. Regular expressions to the rescue

Now that you have extracted the content of the first paragraph, the only thing that remains to finish your Wikipedia scraper is to sanitize the output.

Indeed some Wikipedia references, HTML code, phonetic pronunciation etc. may linger. You might find *regular expressions* handy to get rid of them and obtain pristine text. You will find some useful documentation about regular expressions [here](../../2.python/2.python_advanced/03.Regex/regex.ipynb)

Once you have one of your regex working online, try it in the cell below. 

Hints: 
* Check the `sub()` method documentation.
* Make sure to test urls in different languages. Some may look good but other do not.

In [74]:
# 3 lines
import re; from bs4 import BeautifulSoup
first_paragraph = re.sub(r'\[[^\]]*\]', '', first_paragraph)
print(first_paragraph)

Abraham Lincoln (February 12, 1809 – April 15, 1865) was the 16th president of the United States, serving from 1861 until his assassination in 1865. He led the United States through the American Civil War, defeating the Confederacy and playing a major role in the abolition of slavery.



Overwrite the `get_first_paragraph()` function by applying your regex to the first paragraph before returning it.

In [79]:
# 10 lines
import re; from bs4 import BeautifulSoup; import requests

def get_first_paragraph(wikipedia_url):
    headers = {"User-Agent": "Mozilla/5.0"}
    get_leaders()
    print(wikipedia_url)
    headers = {"User-Agent": "Mozilla/5.0"}
    wiki_text = requests.get(wikipedia_url, headers = headers)
    soup = BeautifulSoup(wiki_text.text, 'html')

    for p in soup.find_all("p"):
        if p.find("b"):
            first_paragraph = p.get_text()
            first_paragraph = re.sub(r'\[[^\]]*\]', '', first_paragraph)

            return first_paragraph

print(first_paragraph)

Abraham Lincoln (February 12, 1809 – April 15, 1865) was the 16th president of the United States, serving from 1861 until his assassination in 1865. He led the United States through the American Civil War, defeating the Confederacy and playing a major role in the abolition of slavery.



Come up with other regexes to capture other patterns and sanitize the outputs completely. Modify your `get_first_paragraph()` function accordingly.

In [106]:
# < 20 lines
import requests

def get_leaders():
    #define the urls
    root_url = "https://country-leaders.onrender.com"
    status_url = root_url + "/status"
    countries_url = root_url + "/countries"
    cookie_url = root_url + "/cookie"
    leaders_url = root_url + "/leaders"
    
    #get the cookies
    req1 = requests.get(cookie_url) 
    cookies = req1.cookies

    #get the countries
    countries = requests.get(countries_url, cookies=cookies).json()

    #loop over them and save their leaders in a dictionary
    data ={}
    for country in countries:
        req2 = requests.get(leaders_url, cookies= cookies, params= {"country" : country})
        data[country] = req2.json()
    return data
#___
import re; from bs4 import BeautifulSoup; import requests

def get_first_paragraph(wikipedia_url):
    headers = {"User-Agent": "Mozilla/5.0"}
    get_leaders()
    print(wikipedia_url)
    headers = {"User-Agent": "Mozilla/5.0"}
    wiki_text = requests.get(wikipedia_url, headers = headers)
    soup = BeautifulSoup(wiki_text.text, 'html')

    for p in soup.find_all("p"):
        if p.find("b"):
            first_paragraph = p.get_text()
            first_paragraph = re.sub(r'\[[^\]]*\]', "", first_paragraph) # removes [references]
            first_paragraph = re.sub(r'\([^)]*(Écouter|listen)[^)]*\)', "", first_paragraph)  # removes phonetic pronunciations
            first_paragraph = re.sub(r'ⓘ', '', first_paragraph)  # removes audio icons
            first_paragraph = re.sub(r'\s{2,}', " ", first_paragraph)  # removes double spaces
            first_paragraph = re.sub(r'\s,', ",", first_paragraph)  # fixes " ,"
            return first_paragraph    
        
leaders_per_country = get_leaders()
wikipedia_url = leaders_per_country["ma"][1]["wikipedia_url"]
first_paragraph = get_first_paragraph(wikipedia_url)
print(first_paragraph)

https://ar.wikipedia.org/wiki/%D8%A7%D9%84%D8%AD%D8%B3%D9%86_%D8%A7%D9%84%D8%AB%D8%A7%D9%86%D9%8A_%D8%A8%D9%86_%D9%85%D8%AD%D9%85%D8%AF
الحَسَن الثاني بِنْ مُحمد بِنْ يوسف العَلوي (9 يوليو 1929 – 23 يوليو 1999) هو ثاني ملوك المملكة المغربية بعد الإستقلال، والملك الثاني والعشرون للمغرب من سلالة العلويين الفيلاليين، تولى حكم المملكة المغربية خلفًا لوالده الملك محمد الخامس في 26 فبراير 1961 وحتى وفاته في 23 يوليو 1999. ينتمي الملك الحسن الثاني إلى السلالة العلوية التي تعود في نسبها إلى الحسن بن علي بن أبي طالب، وتحكم المغرب منذ عام 1666 ميلادية ويلقب الحاكم منهم بأمير المؤمنين.


In [117]:
#first_paragraph only

import re; from bs4 import BeautifulSoup; import requests
def get_first_paragraph(wikipedia_url):
    headers = {"User-Agent": "Mozilla/5.0"}
    print(wikipedia_url)
    headers = {"User-Agent": "Mozilla/5.0"}
    wiki_text = requests.get(wikipedia_url, headers = headers)
    soup = BeautifulSoup(wiki_text.text, 'html')

    for p in soup.find_all("p"):
        if p.find("b"):
            first_paragraph = p.get_text()
            first_paragraph = re.sub(r'\[[^\]]*\]', "", first_paragraph) # removes [references]
            first_paragraph = re.sub(r'\([^)]*(Écouter|listen)[^)]*\)', "", first_paragraph)  # removes phonetic pronunciations
            first_paragraph = re.sub(r'ⓘ', '', first_paragraph)  # removes audio icons
            first_paragraph = re.sub(r'\s{2,}', " ", first_paragraph)  # removes double spaces
            first_paragraph = re.sub(r'\s,', ",", first_paragraph)  # fixes " ,"
            return first_paragraph    

## 3. Putting it all together

Let's go back to your `get_leaders()` function and update it with an *inner* loop over each leader. You will query the url provided and extract the first paragraph using the `get_first_paragraph()` function you just finished. You will then update that `leader`'s dictionary and move on to the next one.

Notice, the rest of the code should not change since you modify the leader's data one by one.

In [ ]:
# < 20 lines
import requests 
def get_leaders():
    
    #urls
    root_url = "https://country-leaders.onrender.com"; status_url = root_url + "/status"; countries_url = root_url + "/countries"; cookie_url = root_url + "/cookie"; leaders_url = root_url + "/leaders"
    
    #get the cookies
    req1 = requests.get(cookie_url) 
    cookies = req1.cookies

    #get the countries
    countries = requests.get(countries_url, cookies=cookies).json()

    #loop over them and save their leaders in a dictionary
    data ={}
    
    for country in countries:
        #get the leaders
        leaders = requests.get(leaders_url, cookies=cookies, params= {"country": country}).json()
                #N.B. : leaders isn't a parameter, it's the API data that we retrieve
        for leader in leaders: #where leader is a dictionary

            #query the url provided and extract the first paragraph using the `get_first_paragraph()` function
            url = leader["wikipedia_url"] #"donne moi la valeur associée à la clé wikipedia_url"
            leader["first_paragraph"] = get_first_paragraph(url)

            #update that `leader`'s dictionary and move on to the next one.
        
        data[country] = leaders #où data est un dico initiallement vide
        #data fait une affection dans un dico : "dans le dico data, à la clé 'fr', stocke la liste des leaders francais"

    return data

In [119]:
# Check the output of your function (2 lines)
import json
leaders_per_country = get_leaders(); print(json.dumps(leaders_per_country,indent=4))

https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
https://fr.wikipedia.org/wiki/Nicolas_Sarkozy
https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand
https://fr.wikipedia.org/wiki/Charles_de_Gaulle
https://fr.wikipedia.org/wiki/Jacques_Chirac
https://fr.wikipedia.org/wiki/Val%C3%A9ry_Giscard_d%27Estaing
https://fr.wikipedia.org/wiki/Georges_Pompidou
https://fr.wikipedia.org/wiki/Adolphe_Thiers
https://fr.wikipedia.org/wiki/Napol%C3%A9on_III
https://fr.wikipedia.org/wiki/Paul_Doumer
https://fr.wikipedia.org/wiki/Alain_Poher
https://fr.wikipedia.org/wiki/Albert_Lebrun
https://fr.wikipedia.org/wiki/Ren%C3%A9_Coty
https://fr.wikipedia.org/wiki/Vincent_Auriol
https://fr.wikipedia.org/wiki/Patrice_de_Mac_Mahon
https://fr.wikipedia.org/wiki/%C3%89mile_Loubet
https://fr.wikipedia.org/wiki/Raymond_Poincar%C3%A9
https://fr.wikipedia.org/wiki/Sadi_Carnot_(homme_d%27%C3%89tat)
https://fr.wikipedia.org/wiki/Alexandre_Millerand
https://fr.wikipedia.org/wiki/Gaston_Doumergue
https://fr.wikipedia.

TypeError: string indices must be integers, not 'str'

Does the function crash in the middle of the loop? Chances are the cookies have expired while looping over the leaders.

Modify your function with an *exception* or check if the `status_code` is a cookie error. In either case, get new cookies and query the api again.

If your code did not crash,

In [120]:
# < 25 lines
import requests

def get_leaders():

    root_url = "https://country-leaders.onrender.com"
    cookie_url = root_url + "/cookie"
    countries_url = root_url + "/countries"
    leaders_url = root_url + "/leaders"

    cookies = requests.get(cookie_url).cookies
    countries = requests.get(countries_url, cookies=cookies).json()

    data = {}

    for country in countries:

        req = requests.get(leaders_url, cookies=cookies, params={"country": country})

        if req.status_code == 403:
            cookies = requests.get(cookie_url).cookies
            req = requests.get(leaders_url, cookies=cookies, params={"country": country})

        leaders = req.json()

        for leader in leaders:
            url = leader["wikipedia_url"]
            leader["first_paragraph"] = get_first_paragraph(url)

        data[country] = leaders

    return data

Check the output of your function again.

In [121]:
# Check the output of your function (1 line)
import json; leaders_per_country = get_leaders(); print(json.dumps(leaders_per_country,indent=4))

https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
https://fr.wikipedia.org/wiki/Nicolas_Sarkozy
https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand
https://fr.wikipedia.org/wiki/Charles_de_Gaulle
https://fr.wikipedia.org/wiki/Jacques_Chirac
https://fr.wikipedia.org/wiki/Val%C3%A9ry_Giscard_d%27Estaing
https://fr.wikipedia.org/wiki/Georges_Pompidou
https://fr.wikipedia.org/wiki/Adolphe_Thiers
https://fr.wikipedia.org/wiki/Napol%C3%A9on_III
https://fr.wikipedia.org/wiki/Paul_Doumer
https://fr.wikipedia.org/wiki/Alain_Poher
https://fr.wikipedia.org/wiki/Albert_Lebrun
https://fr.wikipedia.org/wiki/Ren%C3%A9_Coty
https://fr.wikipedia.org/wiki/Vincent_Auriol
https://fr.wikipedia.org/wiki/Patrice_de_Mac_Mahon
https://fr.wikipedia.org/wiki/%C3%89mile_Loubet
https://fr.wikipedia.org/wiki/Raymond_Poincar%C3%A9
https://fr.wikipedia.org/wiki/Sadi_Carnot_(homme_d%27%C3%89tat)
https://fr.wikipedia.org/wiki/Alexandre_Millerand
https://fr.wikipedia.org/wiki/Gaston_Doumergue
https://fr.wikipedia.

Well done! It took a while however... Let's speed things up. The main *bottleneck* is the loop. We call on the Wikipedia website many times.

You will use the same *session* to call all the wikipedia pages. Check the *Advanced Usage* section of the Requests module's documentation.

Start by modifying the `get_first_paragraph()` function to accept a session parameter and adjust the `get()` method call.

In [125]:
# < 20 lines
import re; from bs4 import BeautifulSoup; import requests
def get_first_paragraph(wikipedia_url, session): #modifier le nombre d'argument
    headers = {"User-Agent": "Mozilla/5.0"}
    print(wikipedia_url)
    headers = {"User-Agent": "Mozilla/5.0"}
    wiki_text = session.get(wikipedia_url, headers = headers) #modification requests change into session
    soup = BeautifulSoup(wiki_text.text, 'html')

    for p in soup.find_all("p"):
        if p.find("b"):
            first_paragraph = p.get_text()
            first_paragraph = re.sub(r'\[[^\]]*\]', "", first_paragraph) # removes [references]
            first_paragraph = re.sub(r'\([^)]*(Écouter|listen)[^)]*\)', "", first_paragraph)  # removes phonetic pronunciations
            first_paragraph = re.sub(r'ⓘ', '', first_paragraph)  # removes audio icons
            first_paragraph = re.sub(r'\s{2,}', " ", first_paragraph)  # removes double spaces
            first_paragraph = re.sub(r'\s,', ",", first_paragraph)  # fixes " ,"
            return first_paragraph   

Modify your `get_leaders()` function to make use of a single session for all the Wikipedia calls.
1. create a `Session` object outside of the loop over countries.
2. pass it to the `get_first_paragraph()` function as an argument.

In [126]:
# <25 lines
import requests

def get_leaders():

    root_url = "https://country-leaders.onrender.com"
    cookie_url = root_url + "/cookie"
    countries_url = root_url + "/countries"
    leaders_url = root_url + "/leaders"

    cookies = requests.get(cookie_url).cookies
    countries = requests.get(countries_url, cookies=cookies).json()
    
    session = requests.Session() #session unique pour wikipédia

    data = {}

    for country in countries:

        req = requests.get(leaders_url, cookies=cookies, params={"country": country})

        if req.status_code == 403:
            cookies = requests.get(cookie_url).cookies
            req = requests.get(leaders_url, cookies=cookies, params={"country": country})

        leaders = req.json()

        for leader in leaders:
            url = leader["wikipedia_url"]
            leader["first_paragraph"] = get_first_paragraph(url,session) #session utilisée

        data[country] = leaders

    return data

Test your new functions.

In [127]:
import json; leaders_per_country = get_leaders(); print(json.dumps(leaders_per_country,indent=4))

https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
https://fr.wikipedia.org/wiki/Nicolas_Sarkozy
https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand
https://fr.wikipedia.org/wiki/Charles_de_Gaulle
https://fr.wikipedia.org/wiki/Jacques_Chirac
https://fr.wikipedia.org/wiki/Val%C3%A9ry_Giscard_d%27Estaing
https://fr.wikipedia.org/wiki/Georges_Pompidou
https://fr.wikipedia.org/wiki/Adolphe_Thiers
https://fr.wikipedia.org/wiki/Napol%C3%A9on_III
https://fr.wikipedia.org/wiki/Paul_Doumer
https://fr.wikipedia.org/wiki/Alain_Poher
https://fr.wikipedia.org/wiki/Albert_Lebrun
https://fr.wikipedia.org/wiki/Ren%C3%A9_Coty
https://fr.wikipedia.org/wiki/Vincent_Auriol
https://fr.wikipedia.org/wiki/Patrice_de_Mac_Mahon
https://fr.wikipedia.org/wiki/%C3%89mile_Loubet
https://fr.wikipedia.org/wiki/Raymond_Poincar%C3%A9
https://fr.wikipedia.org/wiki/Sadi_Carnot_(homme_d%27%C3%89tat)
https://fr.wikipedia.org/wiki/Alexandre_Millerand
https://fr.wikipedia.org/wiki/Gaston_Doumergue
https://fr.wikipedia.

## 4. Saving your hard work

The final step is to save the ``leaders_per_country`` dictionary in the `leaders.json` file using the [json](https://docs.python.org/3/library/json.html) module. Check out the `with` statement.

In [128]:
# 3 lines
import json
with open("leaders.json", "w") as f:
    json.dump(leaders_per_country,f,indent=4)

Make sure the file can be read back. Write the code to read the file. And check the variables are the same.

In [ ]:
# 3 lines
import json
with open("leaders.json", "r") as f: loaded_data = json.load(f)
print(loaded_data == leaders_per_country)

True


Make a function `save(leaders_per_country)` to call this code easily.

In [130]:
# 3 lines
import json
def save(leaders_per_country): 
    with open("leaders.json", "w") as f: json.dump(leaders_per_country, f, indent=4)

In [133]:
# Call the function (1 line)
save(leaders_per_country)

## 5. Tidy things up in a stand-alone python script

Congratulations! You now have a working scraper! However, your code is scattered throughout this notebook along side the tutorials. Hardly production ready...

Copy and paste what you need in a separate `leaders_scraper.py` file.
Make sure it works by calling `python3 leaders_scraper.py`

## (Optional) To go further

If you want to practice scraping, you can read this section and tackle the exercises.

1. Restructure your code by using OOP (see ReadMe).
2. You have noticed the API returns very partial results for country leaders. Many are missing. Overwrite the `get_leaders()` function to get its list from Wikipedia and extract their *personal details* from the frame on the side.

Good luck!